# Week 2 · Day 2 — CTEs (Common Table Expressions)
**Goal:** Write cleaner, more readable multi-step queries by naming intermediate results instead of nesting subqueries.

**Dataset:** Canadian logistics — shipments, drivers, clients, routes, warehouses

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Foundation coming in:** JOINs, GROUP BY grain, subqueries, window functions (Day 1)

---
## Business First Prompt

> *The operations manager wants a multi-step analysis: first identify which drivers are above average in freight revenue, then look at which routes those drivers are working, and finally flag whether those routes are long-haul or short-haul. She adds: "I want to be able to read your query and follow each step — not a wall of nested subqueries."*

Write 3–5 sentences below before touching any data:
- How would you solve this with nested subqueries — and why is it hard to read?
- What does a CTE let you do differently?
- How many steps does this analysis require, and what is each step computing?

**Your answer:**

> Using nested subqueries, I would first calculate total freight per driver, then filter for those above the company average, and finally join that result with route data to classify routes as long-haul or short-haul. However, this approach can become difficult to read and maintain because the logic is buried inside multiple layers of subqueries, making it hard to follow each transformation step.A CTE allows me to break the problem into clearer steps, where each part of the analysis is defined separately and can be easily understood and debugged. This analysis requires three main steps: first, compute total freight revenue per driver, second, calculate the company average and filter for above-average drivers and third, join with route data to identify which routes they operate on and classify those routes based on distance.

---
## Setup — run this first

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('logistics_w2.db')
print('Connected to logistics_w2.db')

Connected to logistics_w2.db


---
## Table Preview

In [4]:
for table in ['clients','warehouses','vehicles','drivers','routes','shipments','shipment_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- clients ---
Columns: ['client_id', 'company', 'city', 'province', 'segment']


,client_id,company,city,province,segment
0,1,Maple Retail Co,Toronto,ON,Enterprise
1,2,Pacific Goods Ltd,Vancouver,BC,Enterprise
2,3,Prairie Supply Inc,Calgary,AB,SMB



--- warehouses ---
Columns: ['warehouse_id', 'name', 'city', 'province', 'capacity_sqft']


,warehouse_id,name,city,province,capacity_sqft
0,1,Toronto Central,Toronto,ON,85000
1,2,Vancouver Hub,Vancouver,BC,72000
2,3,Calgary Depot,Calgary,AB,55000



--- vehicles ---
Columns: ['vehicle_id', 'plate', 'type', 'max_kg']


...

,route_id,origin_wh_id,dest_city,dest_province,distance_km
0,1,1,Montreal,QC,541.0
1,2,1,Ottawa,ON,450.0
2,3,1,Halifax,NS,1800.0



--- shipments ---
Columns: ['shipment_id', 'client_id', 'route_id', 'driver_id', 'vehicle_id', 'ship_date', 'delivery_date', 'status', 'freight_charge']


,shipment_id,client_id,route_id,driver_id,vehicle_id,ship_date,delivery_date,status,freight_charge
0,1001,4,4,3,2,2024-11-23,2024-11-24,Delivered,3375.74
1,1002,2,10,7,1,2024-12-12,None,Delayed,4030.98
2,1003,10,1,9,4,2024-01-16,2024-01-20,Delivered,1162.07



--- shipment_items ---
Columns: ['item_id', 'shipment_id', 'description', 'category', 'weight_kg', 'declared_value']


,item_id,shipment_id,description,category,weight_kg,declared_value
0,1,1001,Electronics,Standard,787.6,321.17
1,2,1002,Office Supplies,Standard,614.3,3832.60
2,3,1002,Clothing,Standard,97.7,2512.06


---
## Concept 1 — What Is a CTE?

A CTE (Common Table Expression) is a named temporary result set you define at the top of a query using `WITH`. You can then reference it by name in the main query — just like a table.

**Syntax:**
```sql
WITH cte_name AS (
    SELECT ...
    FROM ...
    WHERE ...
)
SELECT *
FROM cte_name
WHERE ...
```

**CTE vs Subquery — same result, very different readability:**

```sql
-- Subquery version — hard to follow
SELECT *
FROM (
    SELECT driver_id, SUM(freight_charge) AS total_freight
    FROM shipments
    WHERE status = 'Delivered'
    GROUP BY driver_id
) sub
WHERE total_freight > 10000

-- CTE version — reads like plain English
WITH driver_totals AS (
    SELECT driver_id, SUM(freight_charge) AS total_freight
    FROM shipments
    WHERE status = 'Delivered'
    GROUP BY driver_id
)
SELECT *
FROM driver_totals
WHERE total_freight > 10000
```

**Key rules:**
- CTE is defined before the main SELECT using `WITH`
- The CTE only exists for the duration of that one query
- You can define multiple CTEs by separating them with commas
- Each CTE can reference a CTE defined before it

In [5]:
# EXAMPLE — drivers with total delivered freight above $15000
q = """
WITH driver_totals AS (
    SELECT driver_id,
           SUM(freight_charge) AS total_freight,
           COUNT(*) AS total_shipments
    FROM shipments
    WHERE status = 'Delivered'
    GROUP BY driver_id
)
SELECT d.name,
       d.province,
       dt.total_freight,
       dt.total_shipments
FROM driver_totals dt
JOIN drivers d ON dt.driver_id = d.driver_id
WHERE dt.total_freight > 15000
ORDER BY dt.total_freight DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,province,total_freight,total_shipments
0,Sara Singh,BC,37379.58,15
1,Aisha Diallo,QC,34477.68,15
2,Linda Park,BC,33087.99,12
3,Carlos Rivera,MB,30453.34,12
4,James Leblanc,NS,27825.29,13
5,Mike Okafor,ON,27143.77,13
6,Tom Beaumont,AB,22524.42,9


---
## Drill 1 — Drivers with more than 20 delivered shipments

Use a CTE called `active_drivers` to compute total delivered shipments per driver.  
In the main query, join to drivers to get the name and filter where shipment count > 20.  
Columns: name, province, shipment_count.

In [6]:
q1 = """ WITH delivered_shipments AS ( SELECT  driver_id,   
                                                COUNT(DISTINCT shipment_id) AS shipment_count
                                        FROM shipments 
                                        WHERE status = 'Delivered'
                                        GROUP BY driver_id
                                    )
        SELECT d.name,
                d.province,
                shipment_count
        FROM delivered_shipments ds JOIN drivers d ON ds.driver_id= d.driver_id
        WHERE shipment_count > 20
        

"""
df = pd.read_sql_query(q1, conn)
display(df)

,name,province,shipment_count


---
## Drill 2 — Routes with above-average delivered freight

CTE 1 `route_totals`: total delivered freight per route_id.  
CTE 2 `avg_freight`: the average of those route totals.  
Main query: show route_id, dest_city, total_freight — only routes above the average.  

Hint: two CTEs separated by a comma, second CTE references the first.

In [7]:
q2 = """ WITH total_delivered_freight AS ( SELECT route_id,
                                                SUM(freight_charge) AS freight_total
                                                 FROM shipments
                                                 WHERE status = 'Delivered'
                                                 GROUP BY route_id),
        
         avg_freight AS (   SELECT AVG(freight_total) AS avg_freight
                                FROM total_delivered_freight) 
                                
        SELECT tf.route_id,
                r.dest_city, 
                freight_total
        FROM total_delivered_freight tf JOIN routes r ON tf.route_id = r.route_id CROSS JOIN avg_freight af
        WHERE tf.freight_total > af.avg_freight

"""
df = pd.read_sql_query(q2, conn)
display(df)

,route_id,dest_city,freight_total
0,2,Ottawa,30365.64
1,5,Winnipeg,25405.78
2,6,Edmonton,20203.91
3,10,Moncton,24737.20
4,11,Thunder Bay,23353.47


---
## Drill 3 — Clients with no delivered shipments

CTE `delivered_clients`: distinct client_ids that have at least one delivered shipment.  
Main query: show clients NOT IN that CTE.  
Columns: company, segment, city.

Hint: this replaces the LEFT JOIN + IS NULL pattern with a cleaner NOT IN approach.

In [8]:
q3 = """  WITH delivered_clients AS (   SELECT DISTINCT client_id
                                        FROM shipments
                                        WHERE status = 'Delivered')
        
        SELECT c.company,
                c.segment,
                c.city
        FROM  clients c LEFT JOIN delivered_clients dt ON c.client_id = dt.client_id 
        WHERE c.client_id IS NULL


"""
df = pd.read_sql_query(q3, conn)
display(df)

,company,segment,city


---
## Concept 2 — Chaining Multiple CTEs

You can define multiple CTEs and have each one build on the previous. This is where CTEs become truly powerful — each step is named, readable, and debuggable independently.

```sql
WITH
step_one AS (
    -- first computation
    SELECT ...
),
step_two AS (
    -- builds on step_one
    SELECT ...
    FROM step_one
    WHERE ...
),
step_three AS (
    -- builds on step_two or combines both
    SELECT ...
    FROM step_two
    JOIN step_one ON ...
)
SELECT *
FROM step_three
```

**Debugging tip:** To test a CTE in isolation, temporarily change the main SELECT to `SELECT * FROM step_one` — you can validate each step before combining them.

In [9]:
# EXAMPLE — three-step analysis:
# Step 1: total freight per driver
# Step 2: company-wide average
# Step 3: flag each driver above or below average
q = """
WITH
driver_totals AS (
    SELECT s.driver_id,
           d.name,
           d.province,
           SUM(s.freight_charge) AS total_freight
    FROM shipments s
    JOIN drivers d ON s.driver_id = d.driver_id
    WHERE s.status = 'Delivered'
    GROUP BY s.driver_id, d.name, d.province
),
company_avg AS (
    SELECT AVG(total_freight) AS avg_freight
    FROM driver_totals
)
SELECT dt.name,
       dt.province,
       ROUND(dt.total_freight, 2) AS total_freight,
       ROUND(ca.avg_freight, 2) AS company_avg,
       CASE WHEN dt.total_freight >= ca.avg_freight THEN 'Above' ELSE 'Below' END AS vs_avg
FROM driver_totals dt
CROSS JOIN company_avg ca
ORDER BY dt.total_freight DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,province,total_freight,company_avg,vs_avg
0,Sara Singh,BC,37379.58,24161.2,Above
1,Aisha Diallo,QC,34477.68,24161.2,Above
2,Linda Park,BC,33087.99,24161.2,Above
3,Carlos Rivera,MB,30453.34,24161.2,Above
4,James Leblanc,NS,27825.29,24161.2,Above
5,Mike Okafor,ON,27143.77,24161.2,Above
6,Tom Beaumont,AB,22524.42,24161.2,Below
7,Priya Mehta,ON,12719.59,24161.2,Below
8,Ryan Kowalski,ON,9318.18,24161.2,Below
9,Jean Tremblay,QC,6682.19,24161.2,Below


---
## Drill 4 — Client segment report with Above/Below avg flag

CTE 1 `segment_totals`: total delivered freight and shipment count per client segment.  
CTE 2 `overall_avg`: average freight across all segments.  
Main query: show segment, total_freight, shipment_count, and Above/Below flag.  

Hint: use CROSS JOIN to bring the single-row average into the main query.

In [10]:
q4 = """
        WITH segment_total AS ( SELECT c.segment,
                                COUNT(shipment_id) AS shipment_delivered_count,
                                SUM(freight_charge) AS total_freight      
                                FROM shipments s JOIN clients c ON s.client_id = c.client_id
                                WHERE status = 'Delivered'
                                GROUP BY c.segment),
         overall_avg AS (  SELECT AVG(total_freight) AS avg_segments_freight
                                FROM segment_total
                                )
        SELECT st.segment,
                st.total_freight,
                st.shipment_delivered_count,
                (CASE WHEN st.total_freight > oa.avg_segments_freight THEN 'ABOVE' ELSE 'BELOW' END ) AS above_below_flag
         FROM segment_total st CROSS JOIN overall_avg oa 
       
            

"""
df = pd.read_sql_query(q4, conn)
display(df)

,segment,total_freight,shipment_delivered_count,above_below_flag
0,Enterprise,60683.85,28,BELOW
1,Government,55341.79,22,BELOW
2,SMB,125586.39,59,ABOVE


---
## Drill 5 — Top driver per province using CTEs

CTE 1 `driver_totals`: total delivered freight per driver, including province (join to drivers).  
CTE 2 `province_max`: the maximum total_freight per province.  
Main query: join driver_totals to province_max — show only drivers whose total equals the province max.  
Columns: province, name, total_freight.

In [11]:
q5 = """ 
        WITH driver_totals AS ( SELECT s.driver_id,
                                        d.name,
                                        d.province,
                                        SUM(s.freight_charge) AS per_driver_freight
                                FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                                  WHERE s.status = 'Delivered'
                                    GROUP BY s.driver_id),
            province_max AS (   SELECT province,
                                MAX(s2.freight_charge) AS per_province_max 
                                FROM drivers d1 JOIN shipments s2 ON d1.driver_id = s2.driver_id
                                WHERE s2.status = 'Delivered'
                                GROUP BY province)
        SELECT dt.driver_id,
                dt.name,
                dt. per_driver_freight AS total_freight

        FROM driver_totals dt JOIN province_max pm ON dt.province = pm.province
         WHERE pm.per_province_max = dt.per_driver_freight

"""
df = pd.read_sql_query(q5, conn)
display(df)

,driver_id,name,total_freight


---
## Drill 6 — Proof drill: CTE vs subquery

Rewrite the following subquery version using a CTE instead. Both should return identical results.  
Then observe: which is easier to read and debug?

```sql
-- Subquery version to rewrite:
SELECT *
FROM (
    SELECT route_id,
           COUNT(*) AS total_shipments,
           ROUND(AVG(freight_charge), 2) AS avg_freight
    FROM shipments
    WHERE status = 'Delivered'
    GROUP BY route_id
) sub
WHERE avg_freight > 2000
ORDER BY avg_freight DESC
```

In [12]:
# Subquery version — run this first
q6a = """
SELECT *
FROM (
    SELECT route_id,
           COUNT(*) AS total_shipments,
           ROUND(AVG(freight_charge), 2) AS avg_freight
    FROM shipments
    WHERE status = 'Delivered'
    GROUP BY route_id
) sub
WHERE avg_freight > 2000
ORDER BY avg_freight DESC
"""
df = pd.read_sql_query(q6a, conn)
display(df)

,route_id,total_shipments,avg_freight
0,5,9,2822.86
1,9,7,2792.48
2,7,8,2508.82
3,3,8,2395.27
4,2,13,2335.82
5,11,10,2335.35
6,8,7,2312.19
7,12,6,2078.13
8,6,10,2020.39


In [13]:
# CTE version — rewrite it here, should return same result
q6b = """ WITH route_shipments AS  (   SELECT route_id,
                                                COUNT(shipment_id) AS shipment_count,
                                                ROUND(AVG(freight_charge),2) AS avg_freight_per_route
                                        FROM shipments
                                        WHERE status = 'Delivered'
                                        GROUP BY route_id
                                        )
        SELECT *
        FROM route_shipments
        WHERE avg_freight_per_route > 2000
        ORDER BY avg_freight_per_route DESC
"""
df = pd.read_sql_query(q6b, conn)
display(df)

,route_id,shipment_count,avg_freight_per_route
0,5,9,2822.86
1,9,7,2792.48
2,7,8,2508.82
3,3,8,2395.27
4,2,13,2335.82
5,11,10,2335.35
6,8,7,2312.19
7,12,6,2078.13
8,6,10,2020.39


---
## Concept 3 — CTE + Window Functions

CTEs and window functions are a natural pair. A common pattern: compute window function results in a CTE, then filter on those results in the main query.

**Why this is needed:** You can't filter on a window function result directly in WHERE — the window function runs after WHERE. Wrapping it in a CTE solves this.

```sql
-- Find the top shipment per route — CTE holds the ranked results
WITH ranked AS (
    SELECT shipment_id,
           route_id,
           freight_charge,
           RANK() OVER (PARTITION BY route_id ORDER BY freight_charge DESC) AS rnk
    FROM shipments
    WHERE status = 'Delivered'
)
SELECT shipment_id, route_id, freight_charge
FROM ranked
WHERE rnk = 1
ORDER BY route_id
```

This is cleaner than a nested subquery and easier to extend — you can add more CTEs after `ranked` to enrich the result.

In [14]:
# EXAMPLE — top shipment per route using CTE + RANK()
q = """
WITH ranked AS (
    SELECT s.shipment_id,
           s.route_id,
           s.freight_charge,
           s.driver_id,
           RANK() OVER (PARTITION BY s.route_id ORDER BY s.freight_charge DESC) AS rnk
    FROM shipments s
    WHERE s.status = 'Delivered'
)
SELECT r.shipment_id,
       r.route_id,
       r.freight_charge,
       d.name AS driver_name
FROM ranked r
JOIN drivers d ON r.driver_id = d.driver_id
WHERE r.rnk = 1
ORDER BY r.route_id
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,route_id,freight_charge,driver_name
0,1097,1,2350.93,Linda Park
1,1143,2,4349.29,James Leblanc
2,1080,3,4034.67,Sara Singh
3,1110,4,3771.93,Carlos Rivera
4,1183,5,4476.63,James Leblanc
5,1107,6,4485.35,Carlos Rivera
6,1129,7,3881.14,James Leblanc
7,1035,8,3954.01,Aisha Diallo
8,1021,9,4377.71,Mike Okafor
9,1141,10,4416.37,Linda Park


---
## Drill 7 — Top 2 shipments per client by freight charge

CTE `ranked`: rank each delivered shipment within its client_id by freight_charge descending using RANK().  
Main query: filter where rnk <= 2, join to clients for company name.  
Columns: company, segment, shipment_id, freight_charge, rnk.

In [15]:
q7 = """
          WITH ranked AS (  SELECT client_id,
                                    shipment_id,
                                    freight_charge,
                                    RANK () OVER (PARTITION BY client_id ORDER BY freight_charge DESC) AS rank
                            FROM shipments
                            WHERE status = 'Delivered')
            
            SELECT c.company,
                    c.segment,
                    r.shipment_id,
                    r.freight_charge,
                    r.rank
            FROM ranked r JOIN clients c ON r.client_id = c.client_id
            WHERE rank <= 2
            
"""
df = pd.read_sql_query(q7, conn)
display(df)

,company,segment,shipment_id,freight_charge,rank
0,Maple Retail Co,Enterprise,1064,3233.22,1
1,Maple Retail Co,Enterprise,1029,3229.11,2
2,Pacific Goods Ltd,Enterprise,1080,4034.67,1
3,Pacific Goods Ltd,Enterprise,1024,3969.34,2
4,Prairie Supply Inc,SMB,1021,4377.71,1
5,Prairie Supply Inc,SMB,1118,3022.86,2
6,Atlantic Foods,SMB,1141,4416.37,1
7,Atlantic Foods,SMB,1162,3430.56,2
8,Capital Services,Government,1166,4105.84,1
9,Capital Services,Government,1048,3776.79,2


---
## Drill 8 — Drivers ranked overall and within province, showing top 1 per province

CTE 1 `driver_totals`: total delivered freight per driver, join to drivers for name and province.  
CTE 2 `ranked`: add RANK() OVER (PARTITION BY province ORDER BY total_freight DESC) as province_rank.  
Main query: filter where province_rank = 1.  
Columns: province, name, total_freight.

In [16]:
q8 = """
        WITH driver_totals AS ( SELECT  d.name,
                                        d.province,
                                        s.driver_id,
                                        SUM(s.freight_charge) AS total_freight
                                FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                                WHERE s.status = 'Delivered'
                                GROUP BY s.driver_id
                                    ),
        ranked AS (    SELECT  driver_id, name, province, total_freight,
                                RANK () OVER (PARTITION BY province ORDER BY total_freight DESC) AS rank
                        FROM driver_totals
                        )
        SELECT name,
                province,
                
                total_freight
        FROM  ranked r
        WHERE rank = 1
        
"""
df = pd.read_sql_query(q8, conn)
display(df)

,name,province,total_freight
0,Tom Beaumont,AB,22524.42
1,Sara Singh,BC,37379.58
2,Carlos Rivera,MB,30453.34
3,James Leblanc,NS,27825.29
4,Mike Okafor,ON,27143.77
5,Aisha Diallo,QC,34477.68


---
## Drill 9 — Running total of freight per driver, show only rows where running total crosses $20000

CTE `running`: compute running total of freight_charge per driver ordered by ship_date using SUM() OVER.  
Main query: filter where running_total >= 20000, show the first crossing point per driver.  
Columns: driver_id, ship_date, freight_charge, running_total.

Hint: in the main query, use a second CTE or MIN to find the earliest date where running_total >= 20000 per driver.

In [17]:
q9 = """
        WITH running AS (   SELECT driver_id,
                                    freight_charge,
                                    ship_date,
                                    SUM(freight_charge) OVER (PARTITION BY driver_id ORDER BY ship_DATE) AS running_total
                            FROM shipments
                            WHERE status = 'Delivered'
                            ),
        cross_min_date as ( SELECT driver_id,
                             MIN(ship_date) AS earliest_date
                            FROM running
                            WHERE running_total > 2000
                            GROUP BY driver_id
                            )
    SELECT r.driver_id,
            r.freight_charge,
            r.running_total,
            cmd.earliest_date
    FROM running r JOIN cross_min_date cmd ON r.driver_id = cmd.driver_id
    WHERE running_total > 2000

"""
df = pd.read_sql_query(q9, conn)
display(df)

,driver_id,freight_charge,running_total,earliest_date
0,1,1888.49,6682.19,2024-08-05
1,1,3647.88,4793.70,2024-08-05
2,2,190.52,16550.94,2024-03-12
3,2,376.57,16927.51,2024-03-12
4,2,447.63,17375.14,2024-03-12
...,...,...,...,...
100,10,698.93,7968.05,2024-03-04
101,10,1091.27,9318.18,2024-03-04
102,10,1380.35,5145.77,2024-03-04
103,10,2123.35,7269.12,2024-03-04


---
## Concept 4 — CTE for Grain Separation (replacing the FROM subquery)

The grain separation pattern from Day 5 (subquery in FROM) is much cleaner as a CTE.

```sql
-- Old way: subquery in FROM
SELECT segment, AVG(order_revenue) AS avg_order_value
FROM (
    SELECT c.segment, o.order_id, SUM(oi.quantity * oi.unit_price) AS order_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.segment, o.order_id
) sub
GROUP BY segment

-- CTE way: each step named and readable
WITH order_revenue AS (
    SELECT c.segment, o.order_id, SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.segment, o.order_id
)
SELECT segment, ROUND(AVG(revenue), 2) AS avg_order_value
FROM order_revenue
GROUP BY segment
```

Same result, but now `order_revenue` has a name and you can reference it multiple times or add more CTEs after it.

In [18]:
# EXAMPLE — average freight per shipment per route using CTE grain separation
q = """
WITH shipment_totals AS (
    SELECT s.route_id,
           s.shipment_id,
           SUM(si.declared_value) AS shipment_value
    FROM shipments s
    JOIN shipment_items si ON s.shipment_id = si.shipment_id
    WHERE s.status = 'Delivered'
    GROUP BY s.route_id, s.shipment_id
)
SELECT route_id,
       COUNT(*) AS total_shipments,
       ROUND(AVG(shipment_value), 2) AS avg_value_per_shipment
FROM shipment_totals
GROUP BY route_id
ORDER BY avg_value_per_shipment DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,route_id,total_shipments,avg_value_per_shipment
0,8,7,8551.91
1,11,10,8496.12
2,3,8,8432.66
3,2,13,8222.84
4,6,10,8167.46
5,10,13,8094.71
6,1,8,7817.82
7,7,8,7408.62
8,9,7,6962.67
9,12,6,6716.01


---
## Drill 10 — Average declared value per shipment per client segment

CTE `shipment_values`: total declared_value per shipment (join shipment_items to shipments), include client_id.  
CTE `with_segment`: join to clients to add segment.  
Main query: AVG(shipment_value) per segment.  
Columns: segment, avg_value_per_shipment (rounded 2dp).

In [19]:
q10 = """
        WITH shipment_values AS (   SELECT  s.client_id,
                                            
                                            SUM(si.declared_value) AS total_declared_value
                                    FROM shipments S JOIN shipment_items si ON s.shipment_id = si.shipment_id
                                    GROUP BY s.client_id
                                    
                                ),
        with_segment AS (   SELECT c.segment, sv.total_declared_value
                            FROM shipment_values sv JOIN clients c ON sv.client_id = c.client_id
                             )
        SELECT segment,
                Round(AVG(total_declared_value),2) AS avg
        FROM with_segment 
        GROUP BY segment
"""
df = pd.read_sql_query(q10, conn)
display(df)

,segment,avg
0,Enterprise,157676.25
1,Government,133250.68
2,SMB,163427.80


---
## Drill 11 — Long-haul vs short-haul driver performance

CTE 1 `route_class`: classify each route as 'Long-haul' (distance_km >= 1000) or 'Short-haul' (< 1000).  
CTE 2 `driver_performance`: total delivered freight per driver per route class (join shipments to route_class, join to drivers).  
Main query: show driver name, route_class, total_freight. Sort by route_class, total_freight desc.  
Columns: name, route_class, total_freight.

In [20]:
q11 = """
            WITH route_haul AS ( SELECT route_id,
                                        distance_km,
                                        (CASE WHEN distance_km >= 1000 THEN 'Long Haul' ELSE 'Short Haul' END) AS distance 
                                 FROM routes ),
                driver_performance AS ( SELECT  
                                                d.driver_id,
                                                d.name,
                                                distance,
                                                SUM(s.freight_charge) AS total_freight
                                                
                                    FROM route_haul rh JOIN shipments s ON rh.route_id = s.route_id JOIN drivers d ON s.driver_id = d.driver_id  
                                    GROUP BY d.driver_id, d.name, distance   ) 
SELECT name, 
        distance AS route_class,
        total_freight
FROM driver_performance 

            
"""
df = pd.read_sql_query(q11, conn)
display(df)

,name,route_class,total_freight
0,Jean Tremblay,Long Haul,7158.78
1,Jean Tremblay,Short Haul,21174.32
2,Mike Okafor,Long Haul,5438.44
3,Mike Okafor,Short Haul,39779.55
4,Sara Singh,Long Haul,21062.05
5,Sara Singh,Short Haul,46832.45
6,Tom Beaumont,Long Haul,16963.03
7,Tom Beaumont,Short Haul,33677.00
8,Priya Mehta,Long Haul,12153.65
9,Priya Mehta,Short Haul,28180.42


---
## Drill 12 — Monthly freight trend with running total

CTE 1 `monthly`: total delivered freight per month (use `strftime('%Y-%m', ship_date)` to extract month).  
CTE 2 `with_running`: add a running total of freight ordered by month using SUM() OVER.  
Main query: show month, monthly_freight, running_total. Order by month ascending.  

Hint: strftime('%Y-%m', ship_date) returns '2024-01', '2024-02', etc.

In [21]:
q12 = """
        WITH monthly AS (   SELECT ship_date,
                                SUM(freight_charge) AS total_freight
                            FROM shipments 
                            WHERE status = 'Delivered'
                            GROUP BY strftime('%Y-%m', ship_date)
                            ),

        with_running AS (   SELECT ship_date,
                                    total_freight,
                                    SUM(total_freight) OVER (ORDER BY ship_date) AS running_total
                            FROM monthly 
                    )

        SELECT  strftime('%Y-%m', ship_date) AS ship_date,
                total_freight,
                running_total

        FROM with_running
        ORDER BY strftime('%Y-%m', ship_date) ASC 

"""
df = pd.read_sql_query(q12, conn)
display(df)

,ship_date,total_freight,running_total
0,2024-01,23419.09,23419.09
1,2024-02,7405.26,30824.35
2,2024-03,12486.47,43310.82
3,2024-04,17707.74,61018.56
4,2024-05,23969.77,84988.33
5,2024-06,19492.80,104481.13
6,2024-07,14277.86,118758.99
7,2024-08,27379.60,146138.59
8,2024-09,23872.19,170010.78
9,2024-10,27205.40,197216.18


---
## Block 2 · 11:00am — Applied Challenge

> *The operations manager's full request from the Business First Prompt: identify drivers above the company average in freight revenue, show which routes they're working, and flag each route as long-haul or short-haul. She wants the query structured so each step is readable.*

**Query 1:** Three-CTE pipeline — driver totals → company average → above-average drivers with route info  
**Query 2:** Extend Query 1 to also show each route classified as long-haul / short-haul and the driver's rank within that route class

Write your framing sentence first — name each CTE step and what it computes.

**My CTE plan — step names and what each computes:**

> My first cte will query for all drivers and the total freight per route for delivered orders. My second CTE calculates the company avg summing all freight charges and dividing per shipment id. My third CTE uses CASE WHEN to describe each route by the distance travel seperating lonng haul and short haul.

In [22]:
# Query 1 — above-average drivers with route breakdown
q13 = """
        WITH rev_by_driver AS ( SELECT driver_id,
                                        route_id,
                                        SUM(freight_charge) AS total_freight
                                FROM shipments 
                                WHERE status = 'Delivered'
                                GROUP BY driver_id, route_id),
         
         company_avg AS ( SELECT AVG(freight_charge) AS avg_freight
                          FROM shipments
                            WHERE status = 'Delivered'
                            )
        SELECT driver_id,
                route_id,
                total_freight
        FROM rev_by_driver CROSS JOIN company_avg
        WHERE total_freight > avg_freight
        

"""
df = pd.read_sql_query(q13, conn)
display(df)

,driver_id,route_id,total_freight
0,1,8,3647.88
1,2,1,3275.30
2,2,2,4722.94
3,2,4,4053.31
4,2,6,5464.16
5,2,9,4377.71
6,2,12,2851.47
7,3,3,6449.36
8,3,4,6999.26
9,3,5,6642.42


In [23]:
# Query 2 — add route classification and rank within route class
q14 = """
WITH rev_by_driver AS ( SELECT          driver_id,
                                        route_id,
                                        SUM(freight_charge) AS total_freight
                                FROM shipments 
                                WHERE status = 'Delivered'
                                GROUP BY driver_id, route_id),
         
         company_avg AS ( SELECT AVG(freight_charge) AS avg_freight
                          FROM shipments
                            WHERE status = 'Delivered'
                            ),
        route_classification AS (   SELECT route_id,
                                        ( CASE WHEN distance_km >= 1000 THEN 'Long Haul' ELSE 'Short Haul' END ) AS distance
                                    FROM routes 

        )

        SELECT driver_id,
                rv.route_id,
                total_freight,
                distance AS haul
        FROM rev_by_driver rv CROSS JOIN company_avg ca JOIN route_classification rc ON rv.route_id = rc.route_id
        WHERE total_freight > avg_freight 
       ORDER BY total_freight DESC
        
"""
df = pd.read_sql_query(q14, conn)
display(df)

,driver_id,route_id,total_freight,haul
0,8,2,10453.94,Short Haul
1,9,9,9352.61,Short Haul
2,7,5,8442.13,Long Haul
3,6,6,8098.57,Short Haul
4,9,10,7909.93,Short Haul
5,4,2,7538.12,Short Haul
6,7,11,7536.40,Short Haul
7,3,4,6999.26,Short Haul
8,3,5,6642.42,Long Haul
9,3,3,6449.36,Long Haul


**What does the data say? What would you tell the operations manager?**

> 41 out of 68 orders are above the company avg revenue per

---
## Capstone — Full Operations Dashboard

Build a single query using as many CTEs as needed that produces a complete driver operations report:

| Column | Definition |
|---|---|
| name | Driver name |
| province | Driver province |
| total_shipments | COUNT of delivered shipments |
| total_freight | SUM of freight_charge |
| avg_freight | Average freight per shipment (rounded 2dp) |
| overall_rank | RANK() by total_freight desc |
| province_rank | RANK() by total_freight within province |
| vs_company_avg | 'Above' or 'Below' company-wide avg freight |
| top_route_class | Whether the driver's most-used route class is 'Long-haul' or 'Short-haul' |

Order by overall_rank.

Hint: plan your CTEs before writing. Suggested steps: driver_totals → company_avg → ranked → route_class → driver_route_class.

In [24]:
q_capstone = """ 

WITH driver_totals AS ( SELECT   d.name,
                                d.province,
                                s.driver_id,
                                COUNT(shipment_id) AS delivered_count,
                                 SUM(freight_charge) AS total_freight
                                 FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                                 WHERE s.status = 'Delivered'
                                 GROUP BY d.name, d.province, s.driver_id),

company_avg AS (    SELECT AVG(freight_charge) AS avg_freight
                    FROM shipments
                    WHERE status = 'Delivered'),

province_rank AS ( SELECT driver_id, name, province, total_freight, delivered_count,
            RANK() OVER (PARTITION BY province ORDER BY total_freight DESC ) AS province_rank,
            RANK () OVER (ORDER BY total_freight DESC ) AS overall_rank
            FROM driver_totals 
            ),

top_route AS ( SELECT   s1.driver_id,
                        ( CASE WHEN distance_km >= 1000 THEN 'Long Haul' ELSE 'Short Haul' END ) AS distance,
                        COUNT(*) AS distance_count
                FROM shipments s1 JOIN routes r1 ON s1.route_id = r1.route_id
                GROUP BY  s1.driver_id, CASE WHEN distance_km >= 1000 THEN 'Long Haul' ELSE 'Short Haul' END),
    
rank_by_count AS ( SELECT driver_id,
                         distance,
                         RANK() OVER (PARTITION BY driver_id ORDER BY distance_count DESC) AS route_rank
                         FROM top_route)


SELECT pr.name,
       pr.province,
       pr.delivered_count,
       pr.total_freight,
       ROUND(ca.avg_freight, 2) AS avg_freight,
       pr.overall_rank,
       pr.province_rank,
       CASE WHEN pr.total_freight > ca.avg_freight THEN 'Above' ELSE 'Below' END AS vs_company_avg,
       rb.distance AS top_route_class
FROM province_rank pr
CROSS JOIN company_avg ca
JOIN rank_by_count rb ON pr.driver_id = rb.driver_id
WHERE rb.route_rank = 1
ORDER BY pr.overall_rank
"""
df = pd.read_sql_query(q_capstone, conn)
display(df)

,name,province,delivered_count,total_freight,avg_freight,overall_rank,province_rank,vs_company_avg,top_route_class
0,Sara Singh,BC,15,37379.58,2216.62,1,1,Above,Short Haul
1,Aisha Diallo,QC,15,34477.68,2216.62,2,1,Above,Short Haul
2,Linda Park,BC,12,33087.99,2216.62,3,2,Above,Short Haul
3,Carlos Rivera,MB,12,30453.34,2216.62,4,1,Above,Short Haul
4,James Leblanc,NS,13,27825.29,2216.62,5,1,Above,Short Haul
5,Mike Okafor,ON,13,27143.77,2216.62,6,1,Above,Short Haul
6,Tom Beaumont,AB,9,22524.42,2216.62,7,1,Above,Short Haul
7,Priya Mehta,ON,9,12719.59,2216.62,8,2,Above,Short Haul
8,Ryan Kowalski,ON,8,9318.18,2216.62,9,3,Above,Short Haul
9,Jean Tremblay,QC,3,6682.19,2216.62,10,2,Above,Short Haul


---
## AI Audit · last 10 minutes

Paste your capstone query into Claude or ChatGPT and ask:

1. "Is there a CTE in my query that could be simplified or removed?"
2. "What are the trade-offs between CTEs and subqueries in terms of performance?"
3. "How would I explain the CTE structure of this query to a non-technical manager?"

Record your notes below.

**One thing I'll change:**
> *(write here)*

**One thing I disagree with:**
> *(write here)*

---
## Day 2 Checkpoint — answer before Day 3

Plain English — no SQL needed.

**1. What is a CTE and how does it differ from a subquery? When would you choose one over the other?**

> A CTE (Common Table Expression) is a temporary, named result set that can be referenced within a query. Unlike a subquery, which is typically nested inside another query, a CTE is defined separately at the top, making the logic easier to read and reuse. I would choose a CTE when working with multi-step transformations, complex logic, or when I need to reference the same intermediate result multiple times. Subqueries are more suitable for simpler operations where readability is not a concern.

**2. You need to filter on a window function result. Why can't you put it directly in WHERE, and how does a CTE solve this?**

> You can’t use a window function directly in the WHERE clause because WHERE is evaluated before window functions are computed in SQL’s execution order. This means the result of the window function doesn’t exist yet when WHERE is applied. A CTE solves this by first computing the window function in a separate step, and then allowing you to filter on that result in an outer query where the column is already available.

**3. A colleague says "CTEs are just cosmetic — subqueries do the same thing." What's your response?**

> It’s true that CTEs and subqueries can often produce the same results, so in that sense, CTEs can be seen as cosmetic. However, CTEs provide significant advantages in readability and maintainability by allowing you to break complex logic into clear teps. They also make it easier to debug and reuse intermediate results within the same query. While a subquery might be fine for simple cases, CTEs are generally preferred for multi-step analyses because they make the logic more transparent and easier for others to understand.

**4. In the capstone, you needed to compute a company average and reference it on every driver row. What technique did you use (CROSS JOIN) and why is it needed?**

> A CROSS JOIN is used to attach a single aggregated value, like the company average, to every row in another dataset. Since the average is calculated as a separate result (typically returning one row), a CROSS JOIN ensures that this value is combined with every driver row without filtering or matching conditions. This is needed because we want to compare each driver’s performance against the same company-wide average, and a CROSS JOIN effectively gives that single value across all rows.